# Interactive KNN Explorer

Use the controls to see how K-Nearest Neighbour classifies a new student.

**Left panel — model settings:**
- *k*: number of neighbours to consult.
- *Metric*: how to measure distance (`euclidean`, `manhattan`, `chebyshev`).
- *Weights*: `uniform` (every neighbour votes equally) or `distance`.

**Right panel — query student:**
- *Participation Rate* and *Exam Prep Time* sliders move the `?` query point.

**Checkboxes:**
- *Show k-ring*: dashed circle showing the distance to the k-th nearest neighbour.
- *Show neighbour lines*: segments from the query to each of its k neighbours.
- *Show decision boundary*: colour-fill the regions where KNN predicts A or B.

The scatter plot, prediction, and neighbour breakdown update on every change.


In [3]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, Checkbox, VBox, HBox, Output, HTML
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path
from IPython.display import display, clear_output

%matplotlib inline

# def find_dataset_path() -> Path:
#    candidates = [
#        Path('data') / 'student_grades.csv',
#        Path('..') / 'data' / 'student_grades.csv',
#    ]
#    for candidate in candidates:
#        resolved = candidate.resolve()
#        if resolved.exists():
#            return resolved
#    raise FileNotFoundError('Could not locate data/student_grades.csv from the notebook.')

df = pd.read_csv('student_grades.csv')

FEATURES = ['Participation_Rate', 'Exam_Prep_Time']
LABEL = 'Grade'
CLASS_NAMES = ['B', 'A']
X = df[FEATURES].to_numpy()
y = (df[LABEL] == 'A').astype(int).to_numpy()

X_MIN, X_MAX = 0, 100
Y_MIN, Y_MAX = 0, 160

C_A = '#1E88E5'
C_B = '#F9A825'
C_A_LIGHT = '#BBDEFB'
C_B_LIGHT = '#FFF8E1'
C_RING = '#FF8F00'
C_NEIGH = '#FFC107'
C_CORRECT = '#2E7D32'
C_WARN = '#C62828'
C_TEXT = '#212121'
C_GRID = '#E0E0E0'

desc_style = {'description_width': '120px'}

k_slider = IntSlider(value=5, min=1, max=19, step=1,
                     description='k neighbours:', style=desc_style)
query_x = FloatSlider(value=50, min=0, max=100, step=1,
                      description='Participation %:', style=desc_style,
                      readout_format='.0f')
query_y = FloatSlider(value=80, min=0, max=160, step=1,
                      description='Exam Prep (min):', style=desc_style,
                      readout_format='.0f')
metric_w = Dropdown(options=['euclidean', 'manhattan', 'chebyshev'],
                    value='euclidean', description='Distance:', style=desc_style)
weights_w = Dropdown(options=['uniform', 'distance'],
                     value='uniform', description='Vote weight:', style=desc_style)
ring_cb = Checkbox(value=True, description='Show k-ring', indent=False)
lines_cb = Checkbox(value=True, description='Show lines', indent=False)
boundary_cb = Checkbox(value=False, description='Show boundary', indent=False)

plot_out = Output()
info_out = Output()
model_cache = {}

def get_model(k: int, metric: str, weights: str) -> KNeighborsClassifier:
    key = (k, metric, weights)
    if key not in model_cache:
        model_cache[key] = KNeighborsClassifier(
            n_neighbors=k,
            metric=metric,
            weights=weights,
        ).fit(X, y)
    return model_cache[key]

def weighted_vote_sums(dists: np.ndarray, labels: np.ndarray) -> tuple[float, float]:
    safe = np.maximum(dists, 1e-9)
    weights = 1.0 / safe
    return float(weights[labels == 1].sum()), float(weights[labels == 0].sum())

def refresh(_=None):
    k = k_slider.value
    qx, qy = query_x.value, query_y.value
    metric = metric_w.value
    weights = weights_w.value

    clf = get_model(k, metric, weights)
    dists, idxs = clf.kneighbors([[qx, qy]], n_neighbors=k)
    dists, idxs = dists[0], idxs[0]
    labels = y[idxs]
    n_a = int((labels == 1).sum())
    n_b = int((labels == 0).sum())

    pred_int = int(clf.predict([[qx, qy]])[0])
    pred_label = CLASS_NAMES[pred_int]
    pred_color = C_CORRECT if pred_label == 'A' else C_WARN

    with plot_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.set_xlim(X_MIN - 5, X_MAX + 5)
        ax.set_ylim(Y_MIN - 10, Y_MAX + 10)
        ax.set_xlabel('Assignment Participation Rate (%)', fontsize=11)
        ax.set_ylabel('Exam Preparation Time (min)', fontsize=11)
        ax.set_facecolor('#FAFAFA')
        ax.grid(True, color=C_GRID, linewidth=0.5)

        if boundary_cb.value:
            xx, yy = np.meshgrid(
                np.linspace(X_MIN - 5, X_MAX + 5, 200),
                np.linspace(Y_MIN - 10, Y_MAX + 10, 200),
            )
            zz = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
            cmap_bg = ListedColormap([C_B_LIGHT, C_A_LIGHT])
            ax.contourf(xx, yy, zz, alpha=0.35, cmap=cmap_bg, levels=[-0.5, 0.5, 1.5])
            ax.contour(xx, yy, zz, levels=[0.5], colors=['#9E9E9E'], linewidths=1, linestyles='--')

        a_mask = y == 1
        b_mask = y == 0
        ax.scatter(X[b_mask, 0], X[b_mask, 1], s=120, c=C_B, marker='o',
                   edgecolors='#555555', linewidths=0.8, zorder=3, label='B (below avg)')
        ax.scatter(X[a_mask, 0], X[a_mask, 1], s=140, c=C_A, marker='*',
                   edgecolors='#555555', linewidths=0.8, zorder=3, label='A (above avg)')

        for idx in idxs:
            ax.scatter(X[idx, 0], X[idx, 1], s=260, facecolors='none',
                       edgecolors=C_NEIGH, linewidths=2.5, zorder=4)

        if lines_cb.value:
            for idx in idxs:
                ax.plot([qx, X[idx, 0]], [qy, X[idx, 1]], color='#9E9E9E', linewidth=0.9, zorder=2)

        if ring_cb.value:
            ring = mpatches.Circle((qx, qy), float(dists[-1]), fill=False,
                                   edgecolor=C_RING, linewidth=2, linestyle='--', zorder=5)
            ax.add_patch(ring)

        ax.text(qx, qy, '?', fontsize=16, fontweight='bold', color='black', ha='center', va='center', zorder=7)
        ax.legend(loc='lower right', fontsize=10)
        ax.set_title(f'KNN   k={k}   metric={metric}   weights={weights}', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

    with info_out:
        clear_output(wait=True)
        if weights == 'uniform':
            vote_line = f'Votes - A: {n_a}, B: {n_b}'
        else:
            w_a, w_b = weighted_vote_sums(dists, labels)
            vote_line = f'Weighted votes - A: {w_a:.3f}, B: {w_b:.3f}'

        display(HTML(
            f"<div style='font-family:monospace; font-size:14px; line-height:1.8'>"
            f"<b>Prediction:</b> <span style='color:{pred_color}; font-size:16px; font-weight:bold'>{pred_label}</span>"
            f" &nbsp; | &nbsp; {vote_line}</div>"
        ))

        rows_html = ''
        for rank, (idx, dist) in enumerate(zip(idxs, dists), start=1):
            grade = 'A' if y[idx] == 1 else 'B'
            color = C_A if grade == 'A' else C_B
            rows_html += (
                f"<tr>"
                f"<td style='text-align:center'>{rank}</td>"
                f"<td style='text-align:center'>{df.iloc[idx]['Student_ID']}</td>"
                f"<td style='text-align:center'>{df.iloc[idx]['Exam_Prep_Time']}</td>"
                f"<td style='text-align:center'>{df.iloc[idx]['Participation_Rate']}</td>"
                f"<td style='text-align:center; color:{color}; font-weight:bold'>{grade}</td>"
                f"<td style='text-align:center'>{dist:.2f}</td>"
                f"</tr>"
            )

        display(HTML(
            "<table style='border-collapse:collapse; font-size:13px; width:100%'>"
            "<tr style='background:#f0f0f0'>"
            "<th style='padding:4px 8px'>Rank</th>"
            "<th>Student ID</th><th>Exam Prep</th><th>Participation</th><th>Grade</th><th>Distance</th>"
            f"</tr>{rows_html}</table>"
        ))

for widget in (k_slider, query_x, query_y, metric_w, weights_w, ring_cb, lines_cb, boundary_cb):
    widget.observe(refresh, names='value')

model_panel = VBox([
    HTML("<b style='font-size:13px'>Model settings</b>"),
    k_slider,
    metric_w,
    weights_w,
    ring_cb,
    lines_cb,
    boundary_cb,
])
query_panel = VBox([
    HTML("<b style='font-size:13px'>Query student</b>"),
    query_x,
    query_y,
])

display(HBox([model_panel, query_panel]))
display(plot_out)
display(info_out)
refresh()


Output()

Output()